In [19]:
LARGE='./raw_data/bv-kg-20260617.large'.freeze

(irb): warning: already initialized constant Object::LARGE


"./raw_data/bv-kg-20260617.large"

In [ ]:
puts `head -3 #{LARGE}`

In [ ]:
puts `grep Drug #{LARGE} | head -3 `


In [ ]:
require 'csv'
drugs = {}
sources = {}
CSV.foreach(LARGE, col_sep: "\t", quote_char: '"', liberal_parsing: true, headers: true) do |row|
  if row["type_1"] == "Drug" || row["type_1"] == "Compound"
    sources[row["source_1"]] = 1
  end
  if row["type_2"] == "Drug"  || row["type_2"] == "Compound"
    sources[row["source_2"]] = 1
  end
end

puts sources.keys


# FAILs
FDA 4901d7c1b4e2aef6913d880bfc91240e
Matches nothing by google or Grok.

All others are MeSH

# Map MeSH to PubChem CUI and formal name

In [ ]:
require 'json'
require 'rest-client'

In [20]:

#  THIS CELL MUST BE RUN!!!
#  THIS CELL MUST BE RUN!!!
#  THIS CELL MUST BE RUN!!!
#  THIS CELL MUST BE RUN!!!


meshdrugs = {}
CSV.foreach(LARGE, col_sep: "\t", quote_char: '"', liberal_parsing: true, headers: true) do |row|
  # this is to eliminate duplicates
  if ["Drug", "Compound"].include?(row["type_1"])
    meshdrugs[row["id_1"]] = row["name_1"]
  end
  if ["Drug", "Compound"].include?(row["type_2"])
    meshdrugs[row["id_2"]] = row["name_2"]
  end
end

puts meshdrugs.keys.size
puts "examples"
puts meshdrugs.keys[0..4]



487
examples
D005680
D012978
D020888
C066471
D014635


In [21]:

require 'rest-client'
require 'json'
RATE_LIMIT_SLEEP = 0.34 # ~3 req/sec safe without API key
REQUEST_TIMEOUT = 30    # seconds
URI_PARSER = URI::Parser.new


def fetch_mesh_json(mesh_ui)
  mesh_ui = mesh_ui.strip.upcase
  url = "https://id.nlm.nih.gov/mesh/#{mesh_ui}.json"
  
  begin
    response = RestClient::Request.execute(
      method: :get,
      url: url,
      timeout: REQUEST_TIMEOUT
    )
    
    if response.code == 200
      JSON.parse(response.body)
    else
      warn "Error fetching MeSH #{mesh_ui}: HTTP #{response.code}"
      false
    end
  rescue RestClient::ExceptionWithResponse => e
    puts "MeSH #{mesh_ui} HTTP error: #{e.message} (code #{e.response&.code})"
    false
  rescue => e
    puts "Exception fetching MeSH #{mesh_ui}: #{e.message}"
    false
  end
end

def fetch_pubchem_cids(endpoint, identifier)
  escaped = URI_PARSER.escape(identifier.strip)
  url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/#{endpoint}/#{escaped}/cids/JSON"
  
  begin
    response = RestClient::Request.execute(
      method: :get,
      url: url,
      timeout: REQUEST_TIMEOUT
    )
    
    if response.code == 200
      data = JSON.parse(response.body)
      cids = data.dig('IdentifierList', 'CID') || []
      cids
    else
      # Non-200 (e.g., 404 for no match) treated as no results, not fatal error
      []
    end
  rescue RestClient::NotFound
    [] # Explicit 404 = no match
  rescue RestClient::ExceptionWithResponse => e
    puts "PubChem #{endpoint} #{identifier} HTTP error: #{e.message} (code #{e.response&.code})"
    false
  rescue => e
    puts "Exception fetching PubChem #{endpoint} #{identifier}: #{e.message}"
    false
  end
end

def extract_cas_like_rn(json)
  rn = json['registryNumber']&.strip
  return nil if rn.nil? || rn.empty? || rn == '0'
  
  # Strict CAS format check (ignores UNII/EC)
  if rn =~ /\A\d{1,7}-\d{2}-\d\z/
    rn
  else
    nil
  end
end


# Example
# Example
# warn fetch_mesh_json('D020888')
j =  fetch_mesh_json('D020888')
puts j
puts "\n\n"
puts extract_cas_like_rn(j)
#warn fetch_mesh_json('D012701'); abort
#puts fetch_mesh_json('D012701')


(irb):3: warning: already initialized constant Object::RATE_LIMIT_SLEEP
(irb):3: warning: previous definition of RATE_LIMIT_SLEEP was here
(irb):4: warning: already initialized constant Object::REQUEST_TIMEOUT
(irb):4: warning: previous definition of REQUEST_TIMEOUT was here
(irb):5: warning: already initialized constant Object::URI_PARSER
(irb):4: warning: previous definition of URI_PARSER was here


{"@id"=>"http://id.nlm.nih.gov/mesh/D020888", "@type"=>"http://id.nlm.nih.gov/mesh/vocab#TopicalDescriptor", "http://id.nlm.nih.gov/mesh/vocab#active"=>true, "allowableQualifier"=>["http://id.nlm.nih.gov/mesh/Q000266", "http://id.nlm.nih.gov/mesh/Q000652", "http://id.nlm.nih.gov/mesh/Q000097", "http://id.nlm.nih.gov/mesh/Q000191", "http://id.nlm.nih.gov/mesh/Q000302", "http://id.nlm.nih.gov/mesh/Q000008", "http://id.nlm.nih.gov/mesh/Q000737", "http://id.nlm.nih.gov/mesh/Q000032", "http://id.nlm.nih.gov/mesh/Q000819", "http://id.nlm.nih.gov/mesh/Q000276", "http://id.nlm.nih.gov/mesh/Q000633", "http://id.nlm.nih.gov/mesh/Q000627", "http://id.nlm.nih.gov/mesh/Q000096", "http://id.nlm.nih.gov/mesh/Q000037", "http://id.nlm.nih.gov/mesh/Q000031", "http://id.nlm.nih.gov/mesh/Q000235", "http://id.nlm.nih.gov/mesh/Q000592", "http://id.nlm.nih.gov/mesh/Q000145", "http://id.nlm.nih.gov/mesh/Q000134", "http://id.nlm.nih.gov/mesh/Q000138", "http://id.nlm.nih.gov/mesh/Q000502", "http://id.nlm.nih.go

# Iteration over all mesh terms

In [22]:
STEREO_PREFIX = /\A(?:(?:[LlDd]-)|(?:\([RrSsEeZz\+\-RS]\)-)|(?:\(\+\/\-\)-))+/

def get_pubchem_title(cid)
  return nil unless cid.to_s =~ /^\d+$/
  url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/#{cid}/property/Title/JSON"
  resp = RestClient::Request.execute(method: :get, url: url, timeout: 15)
  return nil unless resp.code == 200
  JSON.parse(resp.body).dig('PropertyTable', 'Properties', 0, 'Title')
rescue
  nil
end

(irb): warning: already initialized constant Object::STEREO_PREFIX


:get_pubchem_title

In [23]:

CSVFILE = LARGE
OUTPUT = "./maps/2026-biovista-drugs.map".freeze
ERROR  = "./maps/2026-biovista-drugs-errors".freeze
URI_PARSER = URI::Parser.new

out = File.open(OUTPUT, "w")
out.write CSV.generate_line(["biovista_meshid","biovista_label","CID","IUPACname"])

error = File.open(ERROR, "w")

meshdrugs.keys.each do |mesh|
    biovistaname = meshdrugs[mesh]
    json = fetch_mesh_json(mesh)
    if json == false
      warn "#{mesh} ERROR mesh_fetch_failed\n"
      error.write "#{mesh} ERROR mesh_fetch_failed\n"
      sleep(RATE_LIMIT_SLEEP)
      next
    end

# Robust extraction of preferred name (label can be string or hash)
    raw_label = json['label']
    if raw_label.is_a?(Hash)
      name = raw_label['@value'] || 'UNKNOWN'
    elsif raw_label.is_a?(String)
      name = raw_label
    else
      name = 'UNKNOWN'
    end
    name = name.strip

    cids = false
    method = 'none'
    cid_list = ''

    
    # 1. Try RegistryID (MeSH UI directly).
    #    PubChem xref/RegistryID for C-prefix Supplementary Chemical Records is
    #    frequently wrong (wrong compound mapped in PubChem's cross-ref DB), so
    #    skip this step for C-prefix MeSH entries and fall through to name search.
    if !mesh.start_with?('C')
      result = fetch_pubchem_cids('xref/RegistryID', mesh)
      if result == false
        warn "Error on CID for #{mesh} - first attempt"
        error.write "#{mesh} ERROR cid fetch failed on attempt 1\n"
      elsif !result.empty?
        cids = result
        method = 'registry_id'
      end
    end

    # 2. Try CAS if available and we don't have cids yet
    if cids == false || cids.empty?
      cas = extract_cas_like_rn(json)
      if cas
        result = fetch_pubchem_cids('xref/RN', cas)
        if result == false
          warn "Error on CID for #{mesh} - second attempt"
          error.write "#{mesh} ERROR cid fetch failed on attempt 2\n"
        elsif !result.empty?
          cids = result
          method = 'cas_rn'
        end
      end
    end

    # 3. Fallback to name search
    if (cids == false || cids.empty?) && name != 'UNKNOWN'
      result = fetch_pubchem_cids('name', name)
      if result == false
          warn "Error on CID for #{mesh} - third attempt"
          error.write "#{mesh} ERROR cid fetch failed on attempt 3\n"
      elsif !result.empty?
        cids = result
        method = 'name_search'
      end
    end

    # 4. Singular fallback: some MeSH names use plural class forms
    #    (e.g. "Docosahexaenoic Acids" → PubChem has "Docosahexaenoic Acid").
    #    NOTE: if you receive new input files, re-check this mapping by looking
    #    for "No PubChem CID found" entries whose name ends in 's' or 'es' —
    #    a manual singular search on PubChem will usually find the correct CID.
    if (cids == false || cids.empty?) && name != 'UNKNOWN' && name.end_with?('s')
      result = fetch_pubchem_cids('name', name[0..-2])
      if result == false
        warn "Error on CID for #{mesh} - fourth attempt (singular form)"
        error.write "#{mesh} ERROR cid fetch failed on attempt 4\n"
      elsif !result.empty?
        cids = result
        method = 'name_search_singular'
        warn "#{mesh}: plural name '#{name}' failed; singular '#{name[0..-2]}' found CID #{cids.first}"
      end
    end

    if cids == false || cids.empty?
      warn  "No PubChem CID found for #{mesh} (#{name})"
      error.write "No PubChem CID found for #{mesh} (#{name})\n"
      next
    end

    cids.each do |cid|
        pubchem_title = get_pubchem_title(cid)
        iupac_label = pubchem_title ? pubchem_title.gsub(STEREO_PREFIX, '') : name
        out.write CSV.generate_line(["http://purl.bioontology.org/ontology/MESH/#{mesh}",biovistaname,cid,iupac_label])
        warn CSV.generate_line(["http://purl.bioontology.org/ontology/MESH/#{mesh}",biovistaname,cid,iupac_label])
    end
end

out.close
error.close

puts "DONE!"
  

    

(irb):1: warning: already initialized constant Object::CSVFILE
(irb):1: warning: previous definition of CSVFILE was here
(irb):2: warning: already initialized constant Object::OUTPUT
(irb):2: warning: previous definition of OUTPUT was here
(irb):3: warning: already initialized constant Object::ERROR
(irb):3: warning: previous definition of ERROR was here
(irb):4: warning: already initialized constant Object::URI_PARSER
(irb):5: warning: previous definition of URI_PARSER was here
http://purl.bioontology.org/ontology/MESH/D005680,gamma-Aminobutyric Acid,119,Gamma-Aminobutyric Acid
http://purl.bioontology.org/ontology/MESH/D012978,Sodium Oxybate,23663870,Sodium Oxybate
http://purl.bioontology.org/ontology/MESH/D020888,Vigabatrin,5665,Vigabatrin
http://purl.bioontology.org/ontology/MESH/C066471,NCS 382,3613485,Ncs 382
http://purl.bioontology.org/ontology/MESH/D014635,Valproic Acid,3121,Valproic Acid
http://purl.bioontology.org/ontology/MESH/D004298,Dopamine,681,Dopamine
http://purl.bioonto

MeSH 4901D7C1B4E2AEF6913D880BFC91240E HTTP error: 404 Not Found (code 404)


4901d7c1b4e2aef6913d880bfc91240e ERROR mesh_fetch_failed
http://purl.bioontology.org/ontology/MESH/D009536,Niacinamide,936,Nicotinamide
http://purl.bioontology.org/ontology/MESH/D014212,Tretinoin,444795,Retinoic Acid
http://purl.bioontology.org/ontology/MESH/D009555,Ninhydrin,10236,Ninhydrin
http://purl.bioontology.org/ontology/MESH/C541932,Ku 0063794,16736978,Ku-0063794
http://purl.bioontology.org/ontology/MESH/D004837,Epinephrine,5816,Epinephrine
http://purl.bioontology.org/ontology/MESH/C085075,CGP 55845A,9954841,Cgp 55845A
http://purl.bioontology.org/ontology/MESH/D003091,Colistin,5311054,Colistin
http://purl.bioontology.org/ontology/MESH/D011729,Pyridostigmine Bromide,7550,Pyridostigmine Bromide
http://purl.bioontology.org/ontology/MESH/D000109,Acetylcholine,187,Acetylcholine
http://purl.bioontology.org/ontology/MESH/D003345,Corticosterone,5753,Corticosterone
http://purl.bioontology.org/ontology/MESH/C004691,colistinmethanesulfonic acid,216258,Colistinmethanesulfonic acid
http://p

DONE!


In [24]:
# ── Biologics fallback ─────────────────────────────────────────────────────
# For drugs that couldn't get a PubChem compound CID (typically proteins,
# antibodies, vaccines, enzymes), look up a PubChem *Substance* SID instead.
# These are written to the map as SUBSTANCE_<SID> so graphing notebooks build
# the correct https://pubchem.ncbi.nlm.nih.gov/substance/<SID> URI.
#
# NOTE for future input files: if new biologics appear in the errors file,
# they will be picked up automatically here on the next run. If PubChem has
# no substance record for a given name, it lands in the biologics-errors file.

require 'uri'
require 'set'

OUTPUT_MAP    = './maps/2026-biovista-drugs.map'
ERROR_FILE    = './maps/2026-biovista-drugs-errors'
BIO_ERROR_OUT = './maps/2026-biovista-drugs-biologics-errors'

def fetch_pubchem_sids(name)
  escaped = URI.encode_www_form_component(name.strip)
  url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug/substance/name/#{escaped}/sids/JSON"
  begin
    response = RestClient::Request.execute(method: :get, url: url, timeout: 30)
    return response.code == 200 ? JSON.parse(response.body).dig('IdentifierList', 'SID') || [] : []
  rescue RestClient::NotFound
    []
  rescue => e
    warn "SID lookup error for #{name}: #{e.message}"
    nil
  end
end

# Load mesh IDs already in the map (avoid duplicates)
existing_mesh = Set.new
CSV.foreach(OUTPUT_MAP, headers: true) do |row|
  existing_mesh << row['biovista_meshid']&.split('/')&.last
end

out = File.open(OUTPUT_MAP, 'a')
bio_err = File.open(BIO_ERROR_OUT, 'w')
found = 0
not_found = []

File.readlines(ERROR_FILE).each do |line|
  m = line.match(/No PubChem CID found for (\S+) \((.+)\)/)
  next unless m
  mesh_id, mesh_name = m[1].strip, m[2].strip
  next if existing_mesh.include?(mesh_id)

  sleep(0.4)
  sids = fetch_pubchem_sids(mesh_name)
  if sids && !sids.empty?
    sid = sids.first
    row = CSV.generate_line(["http://purl.bioontology.org/ontology/MESH/#{mesh_id}", mesh_name, "SUBSTANCE_#{sid}", mesh_name])
    out.write row
    existing_mesh << mesh_id
    warn "BIOLOGIC #{mesh_id} (#{mesh_name}) → SUBSTANCE_#{sid}"
    found += 1
  else
    not_found << "#{mesh_id} (#{mesh_name})"
    bio_err.write "No SID found for #{mesh_id} (#{mesh_name})\n"
    warn "No SID found: #{mesh_id} (#{mesh_name})"
  end
end

out.close
bio_err.close
puts "Biologics added to map: #{found}"
puts "Could not map (see #{BIO_ERROR_OUT}): #{not_found.size}"
not_found.each { |s| puts "  #{s}" }


No SID found: D013482 (Superoxide Dismutase)
No SID found: D008110 (Liver Extracts)
No SID found: C545824 (amino-acid, glucose, and electrolyte solution)
BIOLOGIC D011113 (Polymyxins) → SUBSTANCE_79461918
BIOLOGIC D048271 (Chitosan) → SUBSTANCE_49902906
No SID found: D011112 (Polymyxin B)
No SID found: D007252 (Influenza Vaccines)
BIOLOGIC D000068258 (Bevacizumab) → SUBSTANCE_46504473
No SID found: D016756 (Immunoglobulins, Intravenous)
BIOLOGIC D007371 (Interferon-gamma) → SUBSTANCE_49985128
No SID found: D022242 (Pneumococcal Vaccines)
BIOLOGIC D006493 (Heparin) → SUBSTANCE_135346864
BIOLOGIC D000068818 (Cetuximab) → SUBSTANCE_445933985
BIOLOGIC D000069283 (Rituximab) → SUBSTANCE_496741027
No SID found: D000086663 (COVID-19 Vaccines)
No SID found: D006820 (Hyaluronic Acid)
No SID found: D014807 (Vitamin D)
BIOLOGIC D000068800 (Etanercept) → SUBSTANCE_10099
No SID found: D019904 (Polymethyl Methacrylate)
BIOLOGIC D002364 (Caseins) → SUBSTANCE_176259306
No SID found: D003176 (Complemen

Biologics added to map: 18
Could not map (see ./maps/2026-biovista-drugs-biologics-errors): 53
  D013482 (Superoxide Dismutase)
  D008110 (Liver Extracts)
  C545824 (amino-acid, glucose, and electrolyte solution)
  D011112 (Polymyxin B)
  D007252 (Influenza Vaccines)
  D016756 (Immunoglobulins, Intravenous)
  D022242 (Pneumococcal Vaccines)
  D000086663 (COVID-19 Vaccines)
  D006820 (Hyaluronic Acid)
  D014807 (Vitamin D)
  D019904 (Polymethyl Methacrylate)
  D003176 (Complement C3)
  C065640 (pyruvate dehydrogenase E1alpha subunit)
  D001905 (Botulinum Toxins)
  D019274 (Botulinum Toxins, Type A)
  D001647 (Bile Acids and Salts)
  D000069896 (Transcription Activator-Like Effector Nucleases)
  C091590 (Antral)
  C079420 (Lorenzo's oil)
  D050759 (Cyclin-Dependent Kinase Inhibitor p21)
  D013024 (Soybean Oil)
  C032523 (ribonuclease SPL)
  D015922 (Complement C1q)
  D015320 (Tachykinins)
  D011486 (Protein C)
  D017293 (Protein S)
  D000990 (Antithrombin III)
  D000515 (alpha 1-Antitryp

["D013482 (Superoxide Dismutase)", "D008110 (Liver Extracts)", "C545824 (amino-acid, glucose, and electrolyte solution)", "D011112 (Polymyxin B)", "D007252 (Influenza Vaccines)", "D016756 (Immunoglobulins, Intravenous)", "D022242 (Pneumococcal Vaccines)", "D000086663 (COVID-19 Vaccines)", "D006820 (Hyaluronic Acid)", "D014807 (Vitamin D)", "D019904 (Polymethyl Methacrylate)", "D003176 (Complement C3)", "C065640 (pyruvate dehydrogenase E1alpha subunit)", "D001905 (Botulinum Toxins)", "D019274 (Botulinum Toxins, Type A)", "D001647 (Bile Acids and Salts)", "D000069896 (Transcription Activator-Like Effector Nucleases)", "C091590 (Antral)", "C079420 (Lorenzo's oil)", "D050759 (Cyclin-Dependent Kinase Inhibitor p21)", "D013024 (Soybean Oil)", "C032523 (ribonuclease SPL)", "D015922 (Complement C1q)", "D015320 (Tachykinins)", "D011486 (Protein C)", "D017293 (Protein S)", "D000990 (Antithrombin III)", "D000515 (alpha 1-Antitrypsin)", "D007074 (Immunoglobulin G)", "D007072 (Immunoglobulin D)", "